In [7]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from scipy.sparse.linalg import eigsh
import math


def WS_sparse_pauli_op(N, x, lam, l0, m_lat, g):
    """
    Construct the SparsePauliOp for

        W_S =  (x/2) * sum_{n=0}^{N-2} (X_n X_{n+1} + Y_n Y_{n+1})
             + (1/2) * sum_{n=0}^{N-2} sum_{k=n+1}^{N-1} (N - k - 1 + lam) * Z_n Z_k
             + sum_{n=0}^{N-2} ( N/4 - (1/2) * floor(n/2) + l0*(N - n - 1) ) * Z_n
             + (m_lat/g) * sqrt(x) * sum_{n=0}^{N-1} (-1)^n * Z_n
             + l0**2 * (N - 1) + (1/2) * l0 * N + (1/8) * N**2 + (lam/4) * N

    Args:
        N (int): number of qubits (sites).
        x (float)
        lam (float): lambda
        l0 (float): ell_0
        m_lat (float)
        g (float)

    Returns:
        SparsePauliOp on N qubits.
    """
    terms = []

    def add_1q(op, i, coeff):
        # Qiskit labels are little-endian: qubit 0 is rightmost
        s = ['I'] * N
        s[N - 1 - i] = op
        terms.append((''.join(s), complex(coeff)))

    def add_2q(op, i, j, coeff):
        s = ['I'] * N
        s[N - 1 - i] = op
        s[N - 1 - j] = op
        terms.append((''.join(s), complex(coeff)))

    # (x/2) * sum (X_i X_{i+1} + Y_i Y_{i+1})
    for n in range(N - 1):
        add_2q('X', n, n + 1, x / 2.0)
        add_2q('Y', n, n + 1, x / 2.0)

    # (1/2) * sum_{n<k} (N - k - 1 + lam) * Z_n Z_k
    for n in range(N - 1):
        for k in range(n + 1, N):
            coeff = 0.5 * (N - k - 1 + lam)
            add_2q('Z', n, k, coeff)

    # sum_{n=0}^{N-2} (N/4 - 1/2 floor(n/2) + l0*(N - n - 1)) * Z_n
    for n in range(N - 1):
        coeff = (N / 4.0) - 0.5 * math.ceil(n / 2) + l0 * (N - n - 1)
        add_1q('Z', n, coeff)

    # (m_lat/g) * sqrt(x) * sum_{n=0}^{N-1} (-1)^n * Z_n
    pref = (m_lat / g) * math.sqrt(x)
    for n in range(N):
        add_1q('Z', n, pref * ((-1) ** n))

    # Constant (identity) term
    const = (l0 ** 2) * (N - 1) + 0.5 * l0 * N + 0.125 * (N ** 2) + (lam / 4.0) * N
    terms.append(('I' * N, complex(const)))

    return SparsePauliOp.from_list(terms).simplify()

In [32]:
m = 0.5
g = 0.3
mat = WS_sparse_pauli_op(8, 1. / (g**2), 100., 0., m, g).to_matrix(sparse=True)
eigen_values, eigen_vectors = eigsh(mat, k=254, which="SM")
sorted(eigen_values / 8)

[np.float64(-8.849650348364305),
 np.float64(-5.582568558433829),
 np.float64(-5.212267559089194),
 np.float64(-4.993805794674185),
 np.float64(-4.743123121198832),
 np.float64(-4.532991220165677),
 np.float64(-4.435857397864146),
 np.float64(-4.287219171566654),
 np.float64(-4.1793246089821965),
 np.float64(-4.022184180772481),
 np.float64(-3.8973263320149214),
 np.float64(-3.751626646715996),
 np.float64(-3.6136042407396527),
 np.float64(-3.39980940650844),
 np.float64(-3.1999669175262735),
 np.float64(-2.949188309525616),
 np.float64(-2.7117535492181286),
 np.float64(-1.5723845980629343),
 np.float64(-1.0313801842228913),
 np.float64(-0.8456070723195617),
 np.float64(-0.648186611279428),
 np.float64(-0.5561533871947492),
 np.float64(-0.5505862838425568),
 np.float64(-0.4264259309068362),
 np.float64(-0.1716846319844598),
 np.float64(-0.17084933720511067),
 np.float64(-0.08718045576693371),
 np.float64(0.03505074122328011),
 np.float64(0.04819910807992602),
 np.float64(0.140045025931

In [33]:
eigen_vectors: np.ndarray
np.abs(eigen_vectors.imag).sum(axis=0)

array([1.28749224, 2.02891941, 1.5011039 , 0.55232929, 2.27732876,
       0.07608139, 2.42189847, 0.04126811, 0.3502477 , 1.89854368,
       0.75888247, 0.95978778, 2.25961287, 1.56734772, 1.65062776,
       1.845536  , 1.14527029, 2.36518374, 3.02522841, 3.7121385 ,
       2.41231442, 3.55775538, 0.31026204, 2.74731022, 3.80151659,
       0.04951821, 3.80771932, 3.4960991 , 1.49849416, 2.20423185,
       3.51945858, 2.27416336, 2.81534458, 0.19088967, 1.68783472,
       3.61862603, 2.77299411, 3.64899386, 0.62601392, 3.1628159 ,
       0.30101188, 3.24319868, 3.71001056, 0.02546489, 3.45929602,
       1.48000274, 2.86390409, 2.71481904, 1.01043678, 1.83413284,
       3.50592762, 1.39501748, 1.56154531, 4.03355984, 4.02166816,
       3.08375772, 3.06797663, 1.2775825 , 0.61452261, 3.65736396,
       3.82231096, 3.60239376, 0.03584324, 3.60325109, 3.49287095,
       0.83749585, 3.83724551, 0.95613407, 3.37911473, 3.6170057 ,
       0.33130647, 2.46569156, 1.71084598, 6.76152784, 2.03538